<a href="https://colab.research.google.com/github/Ananya-mandal56/DWBDA_Assignment/blob/main/Assignment_3_Spark_Sales_Category.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3 — Apache Spark: Number of Products Sold in Each Category

**Objective:** Use Apache Spark to calculate the number of products sold in each category.

### Dataset note
The instructor did not provide a sales dataset, so a **synthetic sales dataset** was created for demonstrating the Spark program. It contains 12 sales records across Electronics, Clothing, Stationery and Footwear.

## 1. Install PySpark

In [ ]:
!pip -q install pyspark

## 2. Import libraries and create Spark session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum

spark = (
    SparkSession.builder
    .appName("SalesCategoryAnalysis")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


## 3. Upload and read the sales dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sales_data.csv to sales_data.csv


In [ ]:
sales_df = spark.read.csv(
    "sales_data.csv",
    header=True,
    inferSchema=True
)

sales_df.show()
sales_df.printSchema()

+-------+----------+-----------+--------+----------+
|Sale_ID|   Product|   Category|Quantity|Unit_Price|
+-------+----------+-----------+--------+----------+
|      1|    Laptop|Electronics|       2|     55000|
|      2|     Mouse|Electronics|       5|       800|
|      3|  Keyboard|Electronics|       3|      1500|
|      4|Headphones|Electronics|       4|      2200|
|      5|     Shirt|   Clothing|       8|       900|
|      6|     Jeans|   Clothing|       4|      1800|
|      7|    Jacket|   Clothing|       3|      2500|
|      8|  Notebook| Stationery|      10|       100|
|      9|       Pen| Stationery|      20|        20|
|     10|    Pencil| Stationery|      15|        10|
|     11|     Shoes|   Footwear|       3|      2500|
|     12|   Sandals|   Footwear|       6|       900|
+-------+----------+-----------+--------+----------+

root
 |-- Sale_ID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Quantity: integer (nul

## 4. Calculate number of products sold in each category

In [ ]:
category_sales = (
    sales_df
    .groupBy("Category")
    .agg(
        spark_sum("Quantity").alias("Total_Products_Sold")
    )
    .orderBy(col("Total_Products_Sold").desc())
)

category_sales.show()

+-----------+-------------------+
|   Category|Total_Products_Sold|
+-----------+-------------------+
| Stationery|                 45|
|   Clothing|                 15|
|Electronics|                 14|
|   Footwear|                  9|
+-----------+-------------------+



## 5. Calculate total sales amount by category (additional analysis)

In [ ]:
sales_with_amount = sales_df.withColumn(
    "Sales_Amount",
    col("Quantity") * col("Unit_Price")
)

category_summary = (
    sales_with_amount
    .groupBy("Category")
    .agg(
        spark_sum("Quantity").alias("Total_Products_Sold"),
        spark_sum("Sales_Amount").alias("Total_Sales_Amount")
    )
    .orderBy(col("Total_Products_Sold").desc())
)

category_summary.show()

+-----------+-------------------+------------------+
|   Category|Total_Products_Sold|Total_Sales_Amount|
+-----------+-------------------+------------------+
| Stationery|                 45|              1550|
|   Clothing|                 15|             21900|
|Electronics|                 14|            127300|
|   Footwear|                  9|             12900|
+-----------+-------------------+------------------+



## Conclusion

Apache Spark was used to process the sales dataset and calculate the total number of products sold in each category. The `groupBy()` operation grouped records by product category, while `sum()` calculated the total quantity sold. An additional analysis calculated the total sales amount for each category.

**Dataset note:** Because no sales dataset was provided by the instructor, the dataset used in this assignment is synthetic and is included only for demonstrating the Spark program.

In [ ]:
spark.stop()